# 01 — Data Fetch

Fetch all raw data for the Lead Acquisition Profitability project:
1. **theLook eCommerce** — 5 tables from BigQuery
2. **Insurance lead data** — local CSV
3. **Synthetic spend data** — generated for both tracks

All outputs saved to `data/raw/`.

In [1]:
from google.cloud import bigquery
import pandas as pd
import numpy as np
from pathlib import Path
from dotenv import load_dotenv
import os

# Paths relative to project root (one level up from notebooks/)
PROJECT_ROOT = Path("..").resolve()
load_dotenv(PROJECT_ROOT / ".env")

PROJECT_ID = os.getenv("GCP_PROJECT_ID")
client = bigquery.Client(project=PROJECT_ID)

RAW_DIR = PROJECT_ROOT / "data" / "raw"
THELOOK_DIR = RAW_DIR / "thelook"
LEADS_DIR = RAW_DIR / "leads"
SPEND_DIR = RAW_DIR / "spend"

/Users/riadanas/Desktop/leadgen project ML/.venv/lib/python3.12/site-packages/google/auth/_default.py:114: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


## 1. theLook eCommerce — BigQuery

Fetch all 5 tables from `bigquery-public-data.thelook_ecommerce`.

In [2]:
def fetch_thelook_table(table_name: str) -> pd.DataFrame:
    """Fetch a theLook table from BigQuery."""
    query = f"SELECT * FROM `bigquery-public-data.thelook_ecommerce.{table_name}`"
    df = client.query(query).to_dataframe()
    print(f"{table_name}: {df.shape[0]:,} rows, {df.shape[1]} cols")
    return df

In [3]:
# Fetch all tables
users = fetch_thelook_table("users")              # demographics, traffic_source, location — core acquisition table
orders = fetch_thelook_table("orders")             # order status, dates — conversion outcomes and timing
order_items = fetch_thelook_table("order_items")   # product-level sale prices — revenue and product mix analysis
products = fetch_thelook_table("products")         # catalog with cost + retail price — margin and category analysis

# Events is massive (millions of rows), so we aggregate to user-level summary in SQL
events_query = """
SELECT
    user_id,
    COUNT(*) AS total_events,
    COUNT(DISTINCT session_id) AS session_count,
    COUNTIF(event_type = 'purchase') AS purchase_events,
    COUNTIF(event_type = 'cart') AS cart_events,
    COUNTIF(event_type = 'product') AS product_views,
    MIN(created_at) AS first_event,
    MAX(created_at) AS last_event
FROM `bigquery-public-data.thelook_ecommerce.events`
GROUP BY user_id
"""
events = client.query(events_query).to_dataframe()  # behavioral signals — session depth, cart activity, engagement
print(f"events (aggregated): {events.shape[0]:,} rows, {events.shape[1]} cols")

/Users/riadanas/Desktop/leadgen project ML/.venv/lib/python3.12/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


users: 100,000 rows, 16 cols
orders: 124,903 rows, 9 cols
order_items: 180,962 rows, 11 cols
products: 29,120 rows, 9 cols
events (aggregated): 80,125 rows, 8 cols


In [4]:
# Quick look at each table
for name, df in [("users", users), ("orders", orders), ("order_items", order_items), ("events", events), ("products", products)]:
    print(f"\n--- {name} ---")
    print(f"Shape: {df.shape}")
    print(f"Columns: {list(df.columns)}")
    print(f"Nulls: {df.isnull().sum().sum()}")


--- users ---
Shape: (100000, 16)
Columns: ['id', 'first_name', 'last_name', 'email', 'age', 'gender', 'state', 'street_address', 'postal_code', 'city', 'country', 'latitude', 'longitude', 'traffic_source', 'created_at', 'user_geom']
Nulls: 0

--- orders ---
Shape: (124903, 9)
Columns: ['order_id', 'user_id', 'status', 'gender', 'created_at', 'returned_at', 'shipped_at', 'delivered_at', 'num_of_item']
Nulls: 237556

--- order_items ---
Shape: (180962, 11)
Columns: ['id', 'order_id', 'user_id', 'product_id', 'inventory_item_id', 'status', 'created_at', 'shipped_at', 'delivered_at', 'returned_at', 'sale_price']
Nulls: 344666

--- events ---
Shape: (80125, 8)
Columns: ['user_id', 'total_events', 'session_count', 'purchase_events', 'cart_events', 'product_views', 'first_event', 'last_event']
Nulls: 1

--- products ---
Shape: (29120, 9)
Columns: ['id', 'cost', 'category', 'name', 'brand', 'retail_price', 'department', 'sku', 'distribution_center_id']
Nulls: 26


In [5]:
# Save to parquet
users.to_parquet(THELOOK_DIR / "users.parquet", index=False)
orders.to_parquet(THELOOK_DIR / "orders.parquet", index=False)
order_items.to_parquet(THELOOK_DIR / "order_items.parquet", index=False)
events.to_parquet(THELOOK_DIR / "events_user_summary.parquet", index=False)
products.to_parquet(THELOOK_DIR / "products.parquet", index=False)

print("theLook data saved to data/raw/thelook/")

theLook data saved to data/raw/thelook/


## 2. Insurance Lead Data

Copy the lead CSV to the raw data directory and do a quick sanity check.

In [6]:
# Load and copy to raw directory
leads = pd.read_csv(PROJECT_ROOT / "datasets" / "insurance_leadgen_data.csv")
leads.to_csv(LEADS_DIR / "insurance_leadgen_data.csv", index=False)

print(f"Leads: {leads.shape[0]:,} rows, {leads.shape[1]} cols")
print(f"Columns: {list(leads.columns)}")
leads.head()

Leads: 7,878 rows, 13 cols
Columns: ['Lead ID', 'Lead Status', 'Premium', 'Age', 'Gender', 'Smoker', 'Current Insurance', 'Device Type', 'Keyword', 'Match Type', 'Patrial Postcode', 'COVER FOR', 'Verification Status']


,Lead ID,Lead Status,Premium,Age,Gender,Smoker,Current Insurance,Device Type,Keyword,Match Type,Patrial Postcode,COVER FOR,Verification Status
0,0005F412,Contacted,0.0,31,Female,Yes,no,Desktop,private health insurance belfast,Exact,Bt13,Self,NaN
1,000A5F9D,Contacted,0.0,35,Female,No,no,Smartphone,private medical insurance,Phrase,LU7,Self,NaN
2,00148A0A,Contacted,0.0,46,Male,No,yes private,Smartphone,private health insurance,Broad,E14,Self,NaN
3,002C53E5,No Contact,0.0,53,Female,No,no,Desktop,bupa health insurance,Exact,W2,Self + Partner,NaN
4,005FC597,Contacted,0.0,35,Male,No,no,Smartphone,private healthcare prices,Exact,BH23,Self,NaN


In [7]:
# Lead status distribution
print(leads["Lead Status"].value_counts())
print(f"\nConversion rate (Sold): {(leads['Lead Status'] == 'Sold').mean():.2%}")

Lead Status
Contacted     6280
Quoted         533
Sold           386
Invalid        342
No Contact     337
Name: count, dtype: int64

Conversion rate (Sold): 4.90%


In [8]:
# Key dimensions we'll use for spend generation
print("Traffic dimensions:")
print(f"  Device types: {leads['Device Type'].unique().tolist()}")
print(f"  Match types: {leads['Match Type'].unique().tolist()}")
print(f"  Unique keywords: {leads['Keyword'].nunique()}")
print(f"\nTop 10 keywords:")
print(leads["Keyword"].value_counts().head(10))

Traffic dimensions:
  Device types: ['Desktop', 'Smartphone', 'Tablet']
  Match types: ['Exact', 'Phrase', 'Broad', nan]
  Unique keywords: 270

Top 10 keywords:
Keyword
private healthcare              1326
private medical insurance        836
bupa health insurance            775
bupa private health              524
private medical insurance uk     414
private health care              262
private health insurance         243
health insurance quotes          242
health insurance uk              228
health insurance cover           168
Name: count, dtype: int64


## 3. Synthetic Spend Data

Generate realistic daily marketing spend for both data tracks.

### 3a. Lead track spend
Keyed on `keyword_group` × `device_type` × `match_type` × `date`.

### 3b. theLook track spend
Keyed on `traffic_source` × `date`.

In [9]:
np.random.seed(42)

# --- Lead track spend ---
# Group keywords into themes based on the actual data
def classify_keyword(kw):
    kw = str(kw).lower()
    if any(w in kw for w in ["health", "medical", "hospital"]):
        return "health_insurance"
    elif any(w in kw for w in ["life", "death", "funeral"]):
        return "life_insurance"
    elif any(w in kw for w in ["family", "couple", "partner"]):
        return "family_cover"
    elif any(w in kw for w in ["cheap", "affordable", "low cost", "budget"]):
        return "budget_insurance"
    elif any(w in kw for w in ["private", "premium", "bupa", "axa"]):
        return "private_insurance"
    elif any(w in kw for w in ["compare", "quote", "best"]):
        return "comparison_shopping"
    else:
        return "general_insurance"

# Verify keyword grouping covers the data
leads["keyword_group"] = leads["Keyword"].apply(classify_keyword)
print("Keyword group distribution:")
print(leads["keyword_group"].value_counts())

Keyword group distribution:
keyword_group
health_insurance     7700
private_insurance     148
general_insurance      30
Name: count, dtype: int64


In [10]:
# Generate 12 months of daily spend for the lead track
dates = pd.date_range("2025-01-01", "2025-12-31", freq="D")
keyword_groups = leads["keyword_group"].unique().tolist()
device_types = ["Desktop", "Smartphone", "Tablet"]
match_types = ["Exact", "Phrase", "Broad"]

# Realistic CPC ranges by match type (exact costs more, broad costs less)
cpc_ranges = {"Exact": (2.5, 6.0), "Phrase": (1.5, 4.0), "Broad": (0.8, 2.5)}

# Generate spend at keyword_group × device × match_type × week level
# (daily per combination would be too granular — weekly is more realistic)
weeks = pd.date_range("2025-01-06", "2025-12-29", freq="W-MON")

rows = []
for week in weeks:
    for kg in keyword_groups:
        for device in device_types:
            for match in match_types:
                # Base clicks vary by keyword group popularity
                base_clicks = np.random.randint(10, 80)
                
                # Device adjustment (desktop gets more spend)
                device_mult = {"Desktop": 1.3, "Smartphone": 1.0, "Tablet": 0.5}[device]
                clicks = int(base_clicks * device_mult)
                
                # CPC based on match type
                low, high = cpc_ranges[match]
                cpc = round(np.random.uniform(low, high), 2)
                
                spend = round(clicks * cpc, 2)
                impressions = int(clicks * np.random.uniform(8, 25))  # CTR ~4-12%
                
                rows.append({
                    "week_start": week,
                    "keyword_group": kg,
                    "device_type": device,
                    "match_type": match,
                    "impressions": impressions,
                    "clicks": clicks,
                    "spend": spend,
                    "avg_cpc": cpc,
                })

lead_spend = pd.DataFrame(rows)
print(f"Lead spend: {lead_spend.shape[0]:,} rows")
print(f"Total spend: £{lead_spend['spend'].sum():,.0f}")
print(f"Date range: {lead_spend['week_start'].min()} to {lead_spend['week_start'].max()}")
lead_spend.head()

Lead spend: 1,404 rows
Total spend: £164,308
Date range: 2025-01-06 00:00:00 to 2025-12-29 00:00:00


,week_start,keyword_group,device_type,match_type,impressions,clicks,spend,avg_cpc
0,2025-01-06,health_insurance,Desktop,Exact,1615,79,460.57,5.83
1,2025-01-06,health_insurance,Desktop,Phrase,1417,91,272.09,2.99
2,2025-01-06,health_insurance,Desktop,Broad,376,42,80.22,1.91
3,2025-01-06,health_insurance,Smartphone,Exact,432,39,126.36,3.24
4,2025-01-06,health_insurance,Smartphone,Phrase,551,30,91.20,3.04


In [11]:
# --- theLook track spend ---
# Daily spend by traffic_source
thelook_sources = users["traffic_source"].unique().tolist()
dates = pd.date_range("2024-01-01", "2025-12-31", freq="D")

# Realistic daily spend ranges by source
source_spend_ranges = {
    "Search": (200, 800),
    "Organic": (0, 0),         # No paid spend on organic
    "Display": (100, 400),
    "Facebook": (150, 500),
    "Email": (20, 80),
}

rows = []
for date in dates:
    for source in thelook_sources:
        low, high = source_spend_ranges.get(source, (50, 200))
        if low == 0 and high == 0:
            continue  # Skip organic — no paid spend
        
        spend = round(np.random.uniform(low, high), 2)
        cpc = round(np.random.uniform(0.5, 3.0), 2)
        clicks = int(spend / cpc) if cpc > 0 else 0
        impressions = int(clicks * np.random.uniform(10, 30))
        
        rows.append({
            "date": date,
            "traffic_source": source,
            "impressions": impressions,
            "clicks": clicks,
            "spend": spend,
            "avg_cpc": cpc,
        })

thelook_spend = pd.DataFrame(rows)
print(f"theLook spend: {thelook_spend.shape[0]:,} rows")
print(f"Total spend: ${thelook_spend['spend'].sum():,.0f}")
print(f"Sources: {thelook_spend['traffic_source'].unique().tolist()}")
thelook_spend.head()

theLook spend: 2,924 rows
Total spend: $809,885
Sources: ['Search', 'Facebook', 'Display', 'Email']


,date,traffic_source,impressions,clicks,spend,avg_cpc
0,2024-01-01,Search,5462,254,561.89,2.21
1,2024-01-01,Facebook,6393,252,300.17,1.19
2,2024-01-01,Display,1099,75,167.88,2.23
3,2024-01-01,Email,344,24,57.52,2.37
4,2024-01-02,Search,6282,284,235.97,0.83


In [12]:
# Save spend data
lead_spend.to_csv(SPEND_DIR / "lead_spend.csv", index=False)
thelook_spend.to_csv(SPEND_DIR / "thelook_spend.csv", index=False)

print("Spend data saved to data/raw/spend/")

Spend data saved to data/raw/spend/


## 4. Summary

Quick summary of all raw data fetched.

In [13]:
print("=" * 50)
print("RAW DATA SUMMARY")
print("=" * 50)
print(f"\ntheLook eCommerce (BigQuery):")
print(f"  users:              {users.shape[0]:>8,} rows")
print(f"  orders:             {orders.shape[0]:>8,} rows")
print(f"  order_items:        {order_items.shape[0]:>8,} rows")
print(f"  events (per user):  {events.shape[0]:>8,} rows")
print(f"  products:           {products.shape[0]:>8,} rows")
print(f"\nInsurance Leads:")
print(f"  leads:              {leads.shape[0]:>8,} rows")
print(f"\nSynthetic Spend:")
print(f"  lead_spend:         {lead_spend.shape[0]:>8,} rows")
print(f"  thelook_spend:      {thelook_spend.shape[0]:>8,} rows")
print(f"\nAll data saved to data/raw/")

RAW DATA SUMMARY

theLook eCommerce (BigQuery):
  users:               100,000 rows
  orders:              124,903 rows
  order_items:         180,962 rows
  events (per user):    80,125 rows
  products:             29,120 rows

Insurance Leads:
  leads:                 7,878 rows

Synthetic Spend:
  lead_spend:            1,404 rows
  thelook_spend:         2,924 rows

All data saved to data/raw/
